In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#proteomics = pd.read_csv('./data/proteomics_phenotypes.continuous.no_outliers.residuals.qnorm.txt', sep = '\t', header = 0)
#proteomics = pd.read_csv('./data/olink2_data_renamed.no_outliers.residuals.qnorm.txt', sep = '\t', header = 0) # Not regressed by BMI
proteomics = pd.read_csv('./data/proteomic_new.no_outliers.residuals.qnorm.txt', sep = '\t', header = 0) # double protien set, not regressed by BMI

In [ ]:
proteomics.rename(columns={'id': 'IID'}, inplace=True)

In [ ]:
pa_data = pd.read_csv('./data/cleaned_T1_percentiles_HHregressed_PHEWAS.no_outliers.residuals.qnorm.txt', sep = '\t', header = 0)

In [ ]:
vae_data = pd.read_csv('./data/results_16dVAE/VAE_16_latent_dimensions_phenotypes.no_outliers.residuals.qnorm.txt', sep = '\t', header = 0)

## Analyse PHEWAS

In [ ]:
# Define the desired columns for cardiac phenotypes
desired_columns = ['Mean_T1', 'T1_Standard_Deviation',
       'T1_0.25th_Percentile', 'T1_1th_Percentile', 'T1_25th_Percentile',
       'T1_50th_Percentile', 'T1_75th_Percentile', 'T1_99th_Percentile',
      #  'T1_99.75th_Percentile', 'Mean_T1_regressed_hematocrit',
      #  'T1_Standard_Deviation_regressed_hematocrit',
      #  'T1_0.25th_Percentile_regressed_hematocrit',
      #  'T1_1th_Percentile_regressed_hematocrit',
      #  'T1_25th_Percentile_regressed_hematocrit',
      #  'T1_50th_Percentile_regressed_hematocrit',
      #  'T1_75th_Percentile_regressed_hematocrit',
      #  'T1_99th_Percentile_regressed_hematocrit',
      #  'T1_99.75th_Percentile_regressed_hematocrit',
      #  'Mean_T1_regressed_hematocrit_hypertension_status',
      #  'T1_Standard_Deviation_regressed_hematocrit_hypertension_status',
      #  'T1_0.25th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_1th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_25th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_50th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_75th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_99th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_99.75th_Percentile_regressed_hematocrit_hypertension_status'
]

# Merge data by FID
merged_data = pd.merge(proteomics, pa_data, on=["IID"])

print(len(merged_data))
#merged_data = merged_data.drop(columns=["0", "1"], axis = 1)

# Ensure the desired columns are present
filtered_columns = [col for col in desired_columns if col in merged_data.columns]
if len(filtered_columns) != len(desired_columns):
    missing_columns = set(desired_columns) - set(filtered_columns)
    print(f"Warning: The following columns are missing from pa_data: {missing_columns}")

# Prepare the output file
output_file = "cardiac_pwas_results_T1_percentiles_HHregressed.tsv"
with open(output_file, "w") as f:
    f.write("PA_Phenotype\tProtein\tBeta\tP-Value\tFDR\tSignificant\n")

# Perform PWAS
for pa_phenotype in filtered_columns:
    print(f"Processing phenotype: {pa_phenotype}")

    results = []  # Temporary storage for current phenotype results

    for protein_col in proteomics.columns[2:]:  # Skip FID and IID in proteomics
        X = merged_data[protein_col]
        y = merged_data[pa_phenotype]

        # Combine X and y and drop missing values
        valid_data = pd.concat([X, y], axis=1).dropna()
        if valid_data.shape[0] < 2:  # Ensure sufficient data for regression
            continue

        X = sm.add_constant(valid_data.iloc[:, 0])  # Add intercept
        y = valid_data.iloc[:, 1]

        try:
            model = sm.OLS(y, X)
            result = model.fit()
            results.append({
                "PA_Phenotype": pa_phenotype,
                "Protein": protein_col,
                "Beta": result.params.iloc[1],  # Protein coefficient
                "P-Value": result.pvalues.iloc[1]  # Protein p-value
            })
        except Exception as e:
            print(f"Error processing {pa_phenotype} vs {protein_col}: {e}")

    # Convert results to DataFrame
    if results:
        results_df = pd.DataFrame(results)

        # Apply multiple testing correction
        results_df["FDR"] = multipletests(results_df["P-Value"], method="fdr_bh")[1]
        results_df["Significant"] = results_df["FDR"] < 0.05

        # Append results to the file
        results_df.to_csv(output_file, sep="\t", index=False, header=False, mode="a")

    print(f"Finished processing phenotype: {pa_phenotype}")

print(f"PWAS results saved to {output_file}")


In [ ]:
# Define the desired columns for VAE cardiac phenotypes
desired_columns = [
    "Latent_0","Latent_1","Latent_2","Latent_3","Latent_4","Latent_5","Latent_6","Latent_7","Latent_8","Latent_9","Latent_10","Latent_11","Latent_12","Latent_13","Latent_14","Latent_15"
]

# Merge data by FID
merged_data = pd.merge(proteomics, vae_data, on=["IID"])
#merged_data = merged_data.drop(columns=["FID_y", "FID_x"], axis = 1)

# Ensure the desired columns are present
filtered_columns = [col for col in desired_columns if col in merged_data.columns]
if len(filtered_columns) != len(desired_columns):
    missing_columns = set(desired_columns) - set(filtered_columns)
    print(f"Warning: The following columns are missing from pa_data: {missing_columns}")

# Prepare the output file
output_file = "cardiac_pwas_results_VAE_dimensions_notBMIregressed.tsv"
with open(output_file, "w") as f:
    f.write("PA_Phenotype\tProtein\tBeta\tP-Value\tFDR\tSignificant\n")

# Perform PWAS
for pa_phenotype in filtered_columns:
    print(f"Processing phenotype: {pa_phenotype}")

    results = []  # Temporary storage for current phenotype results

    for protein_col in proteomics.columns[2:]:  # Skip FID and IID in proteomics
        X = merged_data[protein_col]
        y = merged_data[pa_phenotype]

        # Combine X and y and drop missing values
        valid_data = pd.concat([X, y], axis=1).dropna()
        if valid_data.shape[0] < 2:  # Ensure sufficient data for regression
            continue

        X = sm.add_constant(valid_data.iloc[:, 0])  # Add intercept
        y = valid_data.iloc[:, 1]

        try:
            model = sm.OLS(y, X)
            result = model.fit()
            results.append({
                "PA_Phenotype": pa_phenotype,
                "Protein": protein_col,
                "Beta": result.params.iloc[1],  # Protein coefficient
                "P-Value": result.pvalues.iloc[1]  # Protein p-value
            })
        except Exception as e:
            print(f"Error processing {pa_phenotype} vs {protein_col}: {e}")

    # Convert results to DataFrame
    if results:
        results_df = pd.DataFrame(results)

        # Apply multiple testing correction
        results_df["FDR"] = multipletests(results_df["P-Value"], method="fdr_bh")[1]
        results_df["Significant"] = results_df["FDR"] < 0.05

        # Append results to the file
        results_df.to_csv(output_file, sep="\t", index=False, header=False, mode="a")

    print(f"Finished processing phenotype: {pa_phenotype}")

print(f"PWAS results saved to {output_file}")

## Visualize Results

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import re

# Load the PWAS results
#data = pd.read_csv("./data/cardiac_pwas_results_T1_percentiles_HHregressed.tsv", sep="\t")  # Replace with your file path
#data = pd.read_csv("./data/results_16dVAE/cardiac_pwas_results_VAE_dimensions.tsv", sep="\t")  # Replace with your file path
data = pd.read_csv("./data/results_16dVAE/cardiac_pwas_results_VAE_dimensions_notBMIregressed.tsv", sep="\t")  # Replace with your file path
data.columns = ["PA_Phenotype", "Protein", "Beta", "P-Value", "FDR", "Significant"]

# Calculate the significance threshold
num_proteins = data['Protein'].nunique()
num_phenotypes = data['PA_Phenotype'].nunique()
significance_threshold = 0.05 / (num_proteins * num_phenotypes)

# Add a column for -log10(p-value) for better visualization
data['-log10(P-Value)'] = -data['P-Value'].apply(lambda p: np.log10(p) if p > 0 else float('inf'))

#Define the desired cardiac phenotype columns
# desired_columns = ['Mean_T1', 'T1_Standard_Deviation',
#        'T1_0.25th_Percentile', 'T1_1th_Percentile', 'T1_25th_Percentile',
#        'T1_50th_Percentile', 'T1_75th_Percentile', 'T1_99th_Percentile',
#        'T1_99.75th_Percentile'
      #  ,'Mean_T1_regressed_hematocrit',
      #  'T1_Standard_Deviation_regressed_hematocrit',
      #  'T1_0.25th_Percentile_regressed_hematocrit',
      #  'T1_1th_Percentile_regressed_hematocrit',
      #  'T1_25th_Percentile_regressed_hematocrit',
      #  'T1_50th_Percentile_regressed_hematocrit',
      #  'T1_75th_Percentile_regressed_hematocrit',
      #  'T1_99th_Percentile_regressed_hematocrit',
      #  'T1_99.75th_Percentile_regressed_hematocrit',
      #  'Mean_T1_regressed_hematocrit_hypertension_status',
      #  'T1_Standard_Deviation_regressed_hematocrit_hypertension_status',
      #  'T1_0.25th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_1th_Percentile_regressed_hematocrit_hypertension_status',
      #  'T1_25th_Percentile_regressed_hematocrit_hypertension_status',
#       #  'T1_50th_Percentile_regressed_hematocrit_hypertension_status',
#       #  'T1_75th_Percentile_regressed_hematocrit_hypertension_status',
#       #  'T1_99th_Percentile_regressed_hematocrit_hypertension_status',
#       #  'T1_99.75th_Percentile_regressed_hematocrit_hypertension_status'
# ]

desired_columns = [
       "Latent_0","Latent_1","Latent_2","Latent_3","Latent_4","Latent_5","Latent_6","Latent_7","Latent_8","Latent_9","Latent_10","Latent_11","Latent_12","Latent_13","Latent_14","Latent_15"
]

# Ensure the data contains only the desired phenotypes
data = data[data['PA_Phenotype'].isin(desired_columns)]

# Initialize the Plotly figure
fig = go.Figure()

# Sort the phenotypes for consistent plotting
data['PA_Phenotype'] = pd.Categorical(data['PA_Phenotype'], categories=desired_columns, ordered=True)
data.sort_values(by=['PA_Phenotype', 'Protein'], inplace=True)

# Loop through phenotypes and create separate frames
frames = []
for phenotype in desired_columns:
    subset = data[data['PA_Phenotype'] == phenotype]
    annotations = []

    # Annotate significant proteins
    max_log_pval = subset['-log10(P-Value)'].max()  # Dynamic y-axis limit
    for _, row in subset.iterrows():
        if row['P-Value'] < significance_threshold:
            annotations.append(
                dict(
                    x=row['Protein'],
                    y=row['-log10(P-Value)'],
                    text=row['Protein'],
                    showarrow=True,
                    arrowhead=2,
                    ax=0,
                    ay=-30,
                    font=dict(size=14, color="black"),
                    xanchor="center",  # Center text to avoid overlap
                    yanchor="bottom"
                )
            )

    # Add the frame
    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=subset['Protein'],
                    y=subset['-log10(P-Value)'],
                    mode='markers',
                    marker=dict(color='blue'),
                    name=f"{phenotype}",
                ),
                go.Scatter(
                    x=subset['Protein'],
                    y=[-np.log10(significance_threshold)] * len(subset),
                    mode='lines',
                    line=dict(color='red', dash='dash'),
                    name="Significance Threshold"
                )
            ],
            layout=dict(
                annotations=annotations,
                title=f"PWAS for Cardiac Phenotype: {phenotype}",
                yaxis=dict(range=[0, max_log_pval + 5])  # Dynamic y-axis with padding
            )
        )
    )

# Add the first frame data
if frames:  # Ensure frames exist before accessing
    first_frame = frames[0]
    for trace in first_frame.data:
        fig.add_trace(trace)

# Update figure layout
fig.update_layout(
    xaxis_title="Proteins (Alphabetical Order)",
    yaxis_title="-log10(P-Value)",
    title="Dynamic PWAS: Protein-Wide Association Study Across VAE Dimensions",
    xaxis=dict(tickangle=45, showticklabels=True),
    yaxis=dict(showgrid=True, zeroline=True, showline=True),
    height=800,
    width=1200,
    plot_bgcolor="white",
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, dict(frame=dict(duration=1500, redraw=True), fromcurrent=True)]),  # Increased frame duration
                dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
            ]
        )
    ]
)

# Add frames to the figure
fig.frames = frames

# Show the figure
fig.show()


In [ ]:
data.sort_values(by=['-log10(P-Value)', 'Protein'], inplace=True)
top_protiens = data[data["P-Value"]< significance_threshold]
top_protiens = data[data["FDR"]< 0.5]

In [ ]:
# 1. Proteins in phenotypes containing "hypertension"
hypertension_proteins = top_protiens[top_protiens['PA_Phenotype'].str.contains('hypertension', case=False, na=False)]['Protein'].unique()

# 2. Proteins in phenotypes containing "hematocrit" but NOT "hypertension"
hematocrit_only_proteins = top_protiens[
    (top_protiens['PA_Phenotype'].str.contains('hematocrit', case=False, na=False)) &
    (~top_protiens['PA_Phenotype'].str.contains('hypertension', case=False, na=False))
]['Protein'].unique()

# 3. Proteins in phenotypes containing neither "hypertension" nor "hematocrit"
neither_proteins = top_protiens[
    (~top_protiens['PA_Phenotype'].str.contains('hematocrit', case=False, na=False))
]['Protein'].unique()


hypertension_set = set(hypertension_proteins)
hematocrit_only_set = set(hematocrit_only_proteins)
neither_set = set(neither_proteins)

print(f"Hypertension proteins ({len(hypertension_proteins)}): {sorted(list(hypertension_proteins))}")
print(f"Hematocrit only proteins ({len(hematocrit_only_proteins)}): {sorted(list(hematocrit_only_proteins))}")
print(f"Neither proteins ({len(neither_proteins)}): {sorted(list(neither_proteins))}")

In [ ]:
# Overlap between hypertension and hematocrit-only
hyp_hematocrit_overlap = hypertension_set & hematocrit_only_set
print(f"Hypertension ∩ Hematocrit-only: {len(hyp_hematocrit_overlap)} proteins")
if hyp_hematocrit_overlap:
    print(f"  Proteins: {list(hyp_hematocrit_overlap)}")

# Overlap between hypertension and neither
hyp_neither_overlap = hypertension_set & neither_set
print(f"Hypertension ∩ Neither: {len(hyp_neither_overlap)} proteins")
if hyp_neither_overlap:
    print(f"  Proteins: {list(hyp_neither_overlap)}")

# Overlap between hematocrit-only and neither
hematocrit_neither_overlap = hematocrit_only_set & neither_set
print(f"Hematocrit-only ∩ Neither: {len(hematocrit_neither_overlap)} proteins")
if hematocrit_neither_overlap:
    print(f"  Proteins: {list(hematocrit_neither_overlap)}")

# Three-way overlap (should be empty by definition)
three_way_overlap = hypertension_set & hematocrit_only_set & neither_set
print(f"All three sets: {len(three_way_overlap)} proteins")
if three_way_overlap:
    print(f"  Proteins: {list(three_way_overlap)}")

In [ ]:
# Loop through cardiac phenotypes and create individual figures
for phenotype in desired_columns:
    subset = data[data['PA_Phenotype'] == phenotype]

    # Annotate and sort the top 10 most significant proteins
    top_subset = subset.sort_values(by='P-Value').head(10)  # Top 10 proteins
    top_proteins = top_subset['Protein'].tolist()  # Extract top protein names
    subset['Protein'] = pd.Categorical(subset['Protein'], categories=top_proteins, ordered=True)
    subset.sort_values(by='Protein', inplace=True)

    # Annotations for the top 10 proteins
    annotations = [
        dict(
            x=row['Protein'],
            y=row['-log10(P-Value)'],
            text=row['Protein'],
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-30,
            font=dict(size=14, color="black"),
            xanchor="center",
            yanchor="bottom"
        )
        for _, row in top_subset.iterrows()
    ]

    # Initialize the figure for this phenotype
    fig = go.Figure(
        data=[
            go.Scatter(
                x=subset['Protein'],
                y=subset['-log10(P-Value)'],
                mode='markers',
                marker=dict(color='blue'),
                name=f"{phenotype}",
            ),
            go.Scatter(
                x=subset['Protein'],
                y=[-np.log10(significance_threshold)] * len(subset),
                mode='lines',
                line=dict(color='red', dash='dash'),
                name="Significance Threshold"
            )
        ],
        layout=dict(
            annotations=annotations,
            title=f"PWAS for Cardiac Phenotype: {phenotype}",
            xaxis_title="Top Proteins (Ordered by Significance)",
            yaxis_title="-log10(P-Value)",
            xaxis=dict(tickangle=45, showticklabels=True),
            yaxis=dict(showgrid=True, zeroline=True, showline=True, range=[0, subset['-log10(P-Value)'].max() + 5]),
            height=800,
            width=1200,
            plot_bgcolor="white"
        )
    )

    # Save the figure to an HTML file
    fig.write_html(f"pwas_results_cardiac_{phenotype}.html")

    # Show the figure for interactive exploration
    fig.show()



In [ ]:
# Create a grid of subplots (3 rows, 3 columns for 9 phenotypes)
fig, axes = plt.subplots(7, 4, figsize=(18, 30), constrained_layout=True)
axes = axes.flatten()  # Flatten axes array for easy iteration

# Plot each phenotype in its respective subplot
for ax, phenotype in zip(axes, desired_columns):
    subset = data[data['PA_Phenotype'] == phenotype]

    # Sort by P-value and select top 5 proteins
    top_subset = subset.sort_values(by='P-Value').head(5)
    top_subset = top_subset.sort_values(by='-log10(P-Value)', ascending=True)  # For horizontal bar chart

    # Plot horizontal bars
    ax.barh(top_subset['Protein'], top_subset['-log10(P-Value)'], color='blue', alpha=0.7)
    ax.axvline(-np.log10(significance_threshold), color='red', linestyle='--', label='Significance Threshold')

    # Customize subplot
    ax.set_title(f"{phenotype}", fontsize=10)
    ax.set_xlabel("-log10(P-Value)", fontsize=8)
    ax.set_ylabel("Top Proteins", fontsize=8)
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=7)
    ax.legend(fontsize=7, loc='lower right')

# Adjust layout and show the figure
plt.tight_layout()
plt.suptitle("PWAS Results for T1 Metrics", fontsize=16, y=1.02)
plt.show()

In [ ]:
# Initialize the Plotly figure
fig = go.Figure()

frames = []

# Loop through cardiac phenotypes and create frames for each
for phenotype in desired_columns:
    subset = data[data['PA_Phenotype'] == phenotype]

    # Annotate and sort the top 10 most significant proteins
    top_subset = subset.sort_values(by='P-Value').head(10)  # Top 10 proteins
    top_proteins = top_subset['Protein'].tolist()  # Extract top protein names
    subset['Protein'] = pd.Categorical(subset['Protein'], categories=top_proteins, ordered=True)
    subset.sort_values(by='Protein', inplace=True)

    # Annotations for the top 10 proteins
    annotations = [
        dict(
            x=row['Protein'],
            y=row['-log10(P-Value)'],
            text=row['Protein'],
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-30,
            font=dict(size=14, color="black"),
            xanchor="center",
            yanchor="bottom"
        )
        for _, row in top_subset.iterrows()
    ]

    # Add the frame for this phenotype
    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=subset['Protein'],
                    y=subset['-log10(P-Value)'],
                    mode='markers',
                    marker=dict(color='blue'),
                    name=f"{phenotype}",
                ),
                go.Scatter(
                    x=subset['Protein'],
                    y=[-np.log10(significance_threshold)] * len(subset),
                    mode='lines',
                    line=dict(color='red', dash='dash'),
                    name="Significance Threshold"
                )
            ],
            layout=dict(
                annotations=annotations,
                title=f"PWAS for Cardiac Phenotype: {phenotype}",
                yaxis=dict(range=[0, subset['-log10(P-Value)'].max() + 5])
            )
        )
    )

# Add the first frame data
if frames:
    first_frame = frames[0]
    for trace in first_frame.data:
        fig.add_trace(trace)

# Update figure layout
fig.update_layout(
    xaxis_title="Top Proteins (Ordered by Significance)",
    yaxis_title="-log10(P-Value)",
    title="PWAS Results for Cardiac Phenotypes",
    xaxis=dict(tickangle=45, showticklabels=True),
    yaxis=dict(showgrid=True, zeroline=True, showline=True),
    height=800,
    width=1200,
    plot_bgcolor="white",
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, dict(frame=dict(duration=1500, redraw=True), fromcurrent=True)]),
                dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
            ]
        )
    ]
)

# Add frames to the figure
fig.frames = frames

# Save the combined figure to an HTML file
fig.write_html("pwas_results_cardiac_all.html")

# Show the figure for interactive exploration
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your data is in a DataFrame called 'df'
# df = pd.read_csv('your_file.csv')  # or however you load your data

# List of proteins most relevant to fibrosis (you can customize this)
fibrosis_proteins = [
    'col1a1', 'col3a1', 'col1a2',  # Collagen proteins
    'fn1',  # Fibronectin
    'tgfb1', 'tgfb2', 'tgfb3',  # TGF-beta family
    'acta2',  # Alpha-smooth muscle actin
    'mmp2', 'mmp9',  # Matrix metalloproteinases
    'timp1', 'timp2',  # Tissue inhibitors of metalloproteinases
    'ctgf',  # Connective tissue growth factor
    'pdgf', 'pdgfb',  # Platelet-derived growth factor
    'lox', 'loxl2',  # Lysyl oxidase
    'vim',  # Vimentin
    'postn',  # Periostin
    'igfbp2', 'igfbp5',  # Insulin-like growth factor binding proteins
    'serpine1',  # PAI-1
    'plat',  # Tissue plasminogen activator
]

def create_pwas_heatmap(df, filter_type='top_significant', n_proteins=20,
                        min_log10_pval=None, fibrosis_only=False):
    """
    Create a PWAS results heatmap from the input dataframe.

    Parameters:
    -----------
    df : pandas DataFrame
        Input dataframe with columns: PA_Phenotype, Protein, Beta, P-Value, FDR, Significant, -log10(P-Value)
    filter_type : str
        'top_significant': Keep top N most significant proteins
        'threshold': Keep proteins with -log10(P-Value) above threshold
        'all': Keep all proteins
    n_proteins : int
        Number of proteins to keep if filter_type='top_significant'
    min_log10_pval : float
        Minimum -log10(P-Value) threshold if filter_type='threshold'
    fibrosis_only : bool
        If True, only keep proteins in the fibrosis_proteins list
    """

    # Make a copy to avoid modifying original
    data = df.copy()

    # Filter for significant results only
    data = data[data['Significant'] == True].copy()

    # Filter for fibrosis-related proteins if requested
    if fibrosis_only:
        data = data[data['Protein'].str.lower().isin([p.lower() for p in fibrosis_proteins])]
        print(f"Filtered to {len(data['Protein'].unique())} fibrosis-related proteins")

    # Apply filtering based on filter_type
    if filter_type == 'top_significant':
        # Get the max -log10(P-Value) for each protein across all phenotypes
        protein_max_sig = data.groupby('Protein')['-log10(P-Value)'].max().sort_values(ascending=False)
        top_proteins = protein_max_sig.head(n_proteins).index.tolist()
        data = data[data['Protein'].isin(top_proteins)]
        print(f"Keeping top {n_proteins} most significant proteins")

    elif filter_type == 'threshold':
        if min_log10_pval is None:
            min_log10_pval = 10  # Default threshold
        # Keep proteins that have at least one phenotype above threshold
        proteins_above_threshold = data[data['-log10(P-Value)'] >= min_log10_pval]['Protein'].unique()
        data = data[data['Protein'].isin(proteins_above_threshold)]
        print(f"Keeping proteins with -log10(P-Value) >= {min_log10_pval}")

    # Create pivot table for heatmap
    pivot_data = data.pivot_table(
        index='Protein',
        columns='PA_Phenotype',
        values='-log10(P-Value)',
        aggfunc='first'  # In case of duplicates, take first value
    )

    # Sort columns in a logical order (you can customize this)
    column_order = desired_columns

    #   [
    #     'Mean_T1',
    #     'T1_Standard_Deviation',
    #     'T1_0.25th_Percentile',
    #     'T1_1th_Percentile',
    #     'T1_25th_Percentile',
    #     'T1_50th_Percentile',
    #     'T1_75th_Percentile',
    #     'T1_99th_Percentile',
    #     'T1_99.75th_Percentile',
    # ]

    # Only keep columns that exist in the data
    column_order = [col for col in column_order if col in pivot_data.columns]
    pivot_data = pivot_data[column_order]

    # Sort rows by maximum significance
    row_order = pivot_data.max(axis=1).sort_values(ascending=False).index
    pivot_data = pivot_data.loc[row_order]

    # Create the heatmap
    plt.figure(figsize=(12, max(8, len(pivot_data) * 0.4)))

    # Create heatmap with custom formatting
    ax = sns.heatmap(
        pivot_data,
        cmap='Blues',
        annot=False,
        fmt='.1f',
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': '-log10(P-Value)'},
        vmin=0,
        vmax=pivot_data.max().max() if len(pivot_data) > 0 else 1
    )

    # Customize the plot
    plt.title('A) PWAS Results for T1 Metrics', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Phenotype', fontsize=12, fontweight='bold')
    plt.ylabel('Protein', fontsize=12, fontweight='bold')

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.tight_layout()

    return pivot_data, plt.gcf()


In [ ]:
# Example usage:

# Option 1: Top 15 most significant proteins
pivot_table, fig = create_pwas_heatmap(top_protiens, filter_type='top_significant', n_proteins=15)
#plt.savefig('pwas_heatmap_top15.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Option 2: Proteins with -log10(P-Value) >= 15
pivot_table, fig = create_pwas_heatmap(top_protiens, filter_type='threshold', min_log10_pval=15)
plt.show()

In [ ]:
# Option 3: Only fibrosis-related proteins (from the predefined list)
pivot_table, fig = create_pwas_heatmap(top_protiens, fibrosis_only=True, filter_type='all')
plt.show()

In [ ]:
#Option 4: Top significant proteins that are also fibrosis-related
pivot_table, fig = create_pwas_heatmap(top_protiens, fibrosis_only=True, filter_type='top_significant', n_proteins=20)
plt.show()